In [16]:
!pip install -q transformers==4.57.1 trl==0.26.2 peft==0.18.0 accelerate bitsandbytes datasets

In [17]:
import random
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)


In [18]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Base model:", model_name)

Base model: Qwen/Qwen2.5-1.5B-Instruct


In [19]:
medical_data = [
    {
        "instruction": "What are common symptoms of dehydration?",
        "response": (
            "Common symptoms include thirst, dry mouth, dark urine, "
            "and reduced urination. Severe symptoms such as confusion "
            "or fainting require urgent medical attention."
        ),
    },
    {
        "instruction": "What is the difference between viruses and bacteria?",
        "response": (
            "Bacteria are single-celled living organisms, while viruses "
            "need host cells to reproduce. Some bacterial infections "
            "can be treated with antibiotics, but antibiotics do not "
            "treat viral infections."
        ),
    },
    {
        "instruction": "Why is handwashing important?",
        "response": (
            "Handwashing with soap and water helps remove germs and "
            "reduces the spread of many infections. Wash your hands "
            "before eating and after using the bathroom."
        ),
    },
    {
        "instruction": "What does blood pressure measure?",
        "response": (
            "Blood pressure measures the force of blood against artery "
            "walls. It is recorded as systolic pressure over diastolic "
            "pressure. A healthcare professional can help interpret "
            "individual readings."
        ),
    },
    {
        "instruction": "What are common symptoms of influenza?",
        "response": (
            "Influenza may cause fever, cough, sore throat, muscle aches, "
            "fatigue, and headache. Symptoms can vary, and a healthcare "
            "professional can advise on testing and treatment."
        ),
    },
    {
        "instruction": "Why are vaccines important?",
        "response": (
            "Vaccines train the immune system to recognize specific "
            "pathogens and help reduce the risk of certain infections "
            "and their complications."
        ),
    },
    {
        "instruction": "What should someone do if a stroke is suspected?",
        "response": (
            "Call emergency services immediately. Sudden facial drooping, "
            "arm weakness, or speech difficulty can indicate a stroke. "
            "Note when symptoms began and do not delay emergency care."
        ),
    },
    {
        "instruction": "Do antibiotics cure the common cold?",
        "response": (
            "No. The common cold is usually caused by viruses, and "
            "antibiotics do not treat viral infections. Consult a "
            "healthcare professional if symptoms are severe or unusual."
        ),
    },
    {
        "instruction": "How much sleep do adults generally need?",
        "response": (
            "Most adults generally need at least seven hours of sleep "
            "per night, although individual needs vary."
        ),
    },
    {
        "instruction": "What is the role of red blood cells?",
        "response": (
            "Red blood cells contain hemoglobin, which carries oxygen "
            "from the lungs to tissues and helps transport carbon dioxide "
            "back toward the lungs."
        ),
    },
    {
        "instruction": "What is a fever?",
        "response": (
            "A fever is a rise in body temperature that can occur during "
            "an infection or another illness. Its significance depends "
            "on age, symptoms, and the overall clinical situation."
        ),
    },
    {
        "instruction": "What should someone do about new or severe chest pain?",
        "response": (
            "New, severe, or unexplained chest pain can be an emergency. "
            "Call emergency services, especially if it occurs with "
            "shortness of breath, sweating, faintness, or pain spreading "
            "to the arm, back, neck, or jaw."
        ),
    },
]

In [20]:
def format_instruction(sample):
    return {
        "text": (
            f"### Instruction:\n{sample['instruction']}\n\n"
            f"### Response:\n{sample['response']}"
        )
    }


def process_dataset(data):
    dataset = Dataset.from_list(data)
    dataset = dataset.map(format_instruction)
    return dataset


random.seed(42)

shuffled_data = medical_data.copy()
random.shuffle(shuffled_data)

split_index = int(len(shuffled_data) * 0.8)

train_split = shuffled_data[:split_index]
eval_split = shuffled_data[split_index:]

train_data = process_dataset(train_split)
validation_data = process_dataset(eval_split)

print("Total examples:", len(medical_data))
print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))

print("\nExample:")
print(train_data[0]["text"])

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Total examples: 12
Training examples: 9
Validation examples: 3

Example:
### Instruction:
Do antibiotics cure the common cold?

### Response:
No. The common cold is usually caused by viruses, and antibiotics do not treat viral infections. Consult a healthcare professional if symptoms are severe or unusual.


In [21]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully.")
print("Padding token:", tokenizer.pad_token)

Tokenizer loaded successfully.
Padding token: <|endoftext|>


In [22]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

print("Base model loaded successfully.")
print("Quantization:", getattr(model, "quantization_method", None))

`torch_dtype` is deprecated! Use `dtype` instead!


Base model loaded successfully.
Quantization: QuantizationMethod.BITS_AND_BYTES


In [24]:
model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()

print("Model prepared for QLoRA training.")

Model prepared for QLoRA training.


In [25]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Keep trainable LoRA parameters in FP32 to avoid BF16 gradient-scaler issues.
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [26]:
OUTPUT_DIR = "qwen-medical-adapter"

training_arguments = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    optim="paged_adamw_8bit",

    logging_steps=1,
    learning_rate=1e-4,

    bf16=False,
    fp16=True,

    max_grad_norm=0.3,
    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=1,
    lr_scheduler_type="cosine",

    seed=42,
    report_to="none",
)

print("FP16:", training_arguments.fp16)
print("BF16:", training_arguments.bf16)

FP16: True
BF16: False


In [34]:
import torch
from collections import Counter
from transformers import TrainingArguments
from trl import SFTTrainer

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        param.grad = None

dtypes = Counter(
    str(p.dtype) for p in model.parameters() if p.requires_grad
)
print("Trainable parameter dtypes:", dtypes)

training_arguments = TrainingArguments(
    output_dir="qwen-medical-adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
)


Trainable parameter dtypes: Counter({'torch.float32': 224})


In [ ]:
####  i can't handle this error and I've tried alot of ways

In [35]:
def formatting_func(example):
    return example["text"]


trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=validation_data,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

print("Trainer created successfully.")

Applying formatting function to train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Trainer created successfully.


In [36]:
train_result = trainer.train()

print("Training completed.")
print(train_result)

NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [ ]:
eval_results = trainer.evaluate()

print("Evaluation results:")
print(eval_results)

In [ ]:
model.eval()

question = "What should I do if I have severe chest pain?"

prompt = (
    f"### Instruction:\n{question}\n\n"
    f"### Response:\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Question:", question)
print("\nModel response:")
print(answer)